In [3]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


import ast
import concurrent.futures
import glob
import itertools
import joblib
import os
import pickle
import warnings
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator

import cvxpy as cp
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
#from sklearn.cluster import KMeans

from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm
from collections import Counter
from functools import reduce
from pprint import pprint

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)


warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Import dataset and betas

In [7]:
# Dataset
combined_data = pl.read_parquet("./scaled_numeric_combined_df.parquet").sort(["cutoff","fips","shifted_time"])
# Columns to exclude
index_cols = ["fips", "cutoff"]
exclude_cols = ["State_FIPS_Code", "log_rolled_cases.x", "log_rolled_cases.y", "t0.lm", "r.lm"]
outcome_col = "shifted_log_rolled_cases"
feature_cols = list(set(combined_data.columns) - set(index_cols) - set(exclude_cols) - set([outcome_col]))
# Indices
indices = combined_data.select(index_cols).sort(["cutoff","fips"])
cutoff_list = sorted(indices.select(["cutoff"]).unique().to_pandas().values.reshape(-1))
fips_list = sorted(indices.select(["fips"]).unique().to_pandas().values.reshape(-1))
lambda_exps = range(-2, 5)

In [9]:
cutoff_list

[51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 208,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 220,
 221,
 222,
 223,
 224,
 225,

In [8]:
combined_data.head()

fips,State_FIPS_Code,log_rolled_cases.x,shifted_time,log_rolled_cases.y,V1,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,…,Variant % B.1.617.3,Variant % B.1.621,Variant % B.1.621.1,Variant % B.1.626,Variant % B.1.628,Variant % B.1.637,Variant % BA.1.1,Variant % BA.2,Variant % BA.2.12.1,Variant % BA.2.75,Variant % BA.2.75.2,Variant % BA.4,Variant % BA.4.6,Variant % BA.5,Variant % BA.5.2.6,Variant % BF.11,Variant % BF.7,Variant % BN.1,Variant % BQ.1,Variant % BQ.1.1,Variant % CH.1.1,Variant % FD.2,Variant % Other,Variant % P.1,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.16,Variant % XBB.1.5,Variant % XBB.1.5.1,Variant % XBB.1.9.1,Variant % XBB.1.9.2,Variant % XBB.2.3,cutoff,t0.lm,r.lm,shifted_log_rolled_cases
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64
6085,6,3.513675,-0.5,3.513675,-0.423156,-0.013292,-0.089641,0.001595,0.177779,0.173636,0.177994,0.083435,0.128384,0.361561,0.100795,0.175218,0.179688,0.137068,0.114342,0.168947,0.121585,0.145852,0.180701,0.131206,0.083488,0.184819,-0.140329,-0.091047,-0.038637,-0.020828,0.361561,-0.051984,-0.029949,-0.073862,-0.088773,-0.01208,…,-0.001177,-0.008002,-0.002919,-0.000839,-0.014696,-0.00884,-0.079609,-0.04548,-0.05064,-0.047448,-0.022975,-0.061483,-0.077053,-0.138342,-0.038974,-0.053896,-0.044289,-0.042387,-0.056376,-0.06653,-0.045461,-0.004132,0.11617,-0.031213,0.231225,0.31534,-0.045977,-0.004622,-0.044236,-0.015737,-0.009342,-0.006405,-0.007827,51,30.345181,0.178769,-0.000156
6085,6,3.692445,0.5,3.513675,-0.423156,-0.013292,-0.089641,0.001595,0.177779,0.173636,0.177994,0.083435,0.128384,0.361561,0.100795,0.175218,0.179688,0.137068,0.114342,0.168947,0.121585,0.145852,0.180701,0.131206,0.083488,0.184819,-0.140329,-0.091047,-0.038637,-0.020828,0.361561,-0.051984,-0.029949,-0.073862,-0.088773,-0.01208,…,-0.001177,-0.008002,-0.002919,-0.000839,-0.014696,-0.00884,-0.079609,-0.04548,-0.05064,-0.047448,-0.022975,-0.061483,-0.077053,-0.138342,-0.038974,-0.053896,-0.044289,-0.042387,-0.056376,-0.06653,-0.045461,-0.004132,0.11617,-0.031213,0.231225,0.31534,-0.045977,-0.004622,-0.044236,-0.015737,-0.009342,-0.006405,-0.007827,51,30.345181,0.178769,0.040961
36119,36,4.315582,-0.5,4.315582,0.103029,0.043325,0.04981,-0.004316,0.083367,0.090947,0.090824,0.04404,0.072736,0.391958,0.048521,0.107483,0.083724,0.074802,0.072031,0.053212,0.040544,0.104242,-0.031047,0.033512,0.13456,0.129816,-0.116736,-0.081613,-0.004035,-0.019004,0.391958,-0.040329,-0.023763,-0.073862,-0.024025,-0.007493,…,-0.001177,-0.008002,-0.002919,-0.000839,-0.014696,-0.00884,-0.079609,-0.04548,-0.05064,-0.047448,-0.022975,-0.061483,-0.077053,-0.138342,-0.038974,-0.053896,-0.044289,-0.042387,-0.056376,-0.06653,-0.045461,-0.004132,0.340294,-0.031213,-0.076036,0.135403,-0.045977,-0.004622,-0.044236,-0.015737,-0.009342,-0.006405,-0.007827,51,30.526731,0.221616,-0.000156
36119,36,4.537197,0.5,4.315582,0.103029,0.043325,0.04981,-0.004316,0.083367,0.090947,0.090824,0.04404,0.072736,0.391958,0.048521,0.107483,0.083724,0.074802,0.072031,0.053212,0.040544,0.104242,-0.031047,0.033512,0.13456,0.129816,-0.116736,-0.081613,-0.004035,-0.019004,0.391958,-0.040329,-0.023763,-0.073862,-0.024025,-0.007493,…,-0.001177,-0.008002,-0.002919,-0.000839,-0.014696,-0.00884,-0.079609,-0.04548,-0.05064,-0.047448,-0.022975,-0.061483,-0.077053,-0.138342,-0.038974,-0.053896,-0.044289,-0.042387,-0.056376,-0.06653,-0.045461,-0.004132,0.340294,-0.031213,-0.076036,0.135403,-0.045977,-0.004622,-0.044236,-0.015737,-0.009342,-0.006405,-0.007827,51,30.526731,0.221616,0.050815
53033,53,4.759729,-0.5,4.759729,0.447

In [3]:
len(feature_cols)

470

In [4]:

cutoff_list = range(500,510)
fips_list = sorted(indices.select(["fips"]).unique().to_pandas().values.reshape(-1))
lambda_exps = range(1, 2)


In [5]:
fips_list

[1001,
 1003,
 1005,
 1007,
 1009,
 1011,
 1013,
 1015,
 1017,
 1019,
 1021,
 1023,
 1025,
 1027,
 1029,
 1031,
 1033,
 1035,
 1037,
 1039,
 1041,
 1043,
 1045,
 1047,
 1049,
 1051,
 1053,
 1055,
 1057,
 1059,
 1061,
 1063,
 1065,
 1067,
 1069,
 1071,
 1073,
 1075,
 1077,
 1079,
 1081,
 1083,
 1085,
 1087,
 1089,
 1091,
 1093,
 1095,
 1097,
 1099,
 1101,
 1103,
 1105,
 1107,
 1109,
 1111,
 1113,
 1115,
 1117,
 1119,
 1121,
 1123,
 1125,
 1127,
 1129,
 1131,
 1133,
 2013,
 2016,
 2020,
 2050,
 2068,
 2070,
 2090,
 2100,
 2110,
 2122,
 2130,
 2150,
 2158,
 2170,
 2180,
 2185,
 2188,
 2195,
 2198,
 2220,
 2230,
 2240,
 2261,
 2275,
 2290,
 2997,
 2998,
 4001,
 4003,
 4005,
 4007,
 4009,
 4011,
 4012,
 4013,
 4015,
 4017,
 4019,
 4021,
 4023,
 4025,
 4027,
 5001,
 5003,
 5005,
 5007,
 5009,
 5011,
 5013,
 5015,
 5017,
 5019,
 5021,
 5023,
 5025,
 5027,
 5029,
 5031,
 5033,
 5035,
 5037,
 5039,
 5041,
 5043,
 5045,
 5047,
 5049,
 5051,
 5053,
 5055,
 5057,
 5059,
 5061,
 5063,
 5065,
 5067,

### Load betas
- Load a later day so that `beta_{t}` and `beta_{-t,c}` don't differ that much

In [6]:
t = 600
c = 2240

In [7]:
stage1_model = joblib.load("./stage1_lasso/best_lasso_model_cutoff={}.pkl".format(t))
beta = stage1_model.coef_
intercept = stage1_model.intercept_
stage1_beta_with_intercept = np.concatenate((beta, [intercept]))


In [8]:
beta.shape, intercept

((470,), -0.0005376366627267531)

In [9]:
stage1_beta_with_intercept.shape

(471,)

In [10]:
stage1_model.n_features_in_

470

### Test `X_{t,c}`

In [11]:
os.makedirs("./stage2_betas", exist_ok=True)
for lambda_exp in lambda_exps:
    os.makedirs("./stage2_betas/lambda_exp={}".format(int(lambda_exp)), exist_ok=True)

In [1]:
lambda_reg = 1.0  # Regularization parameter

def slice_data(combined_data, t, c):
    x_tc = combined_data.filter(pl.col("cutoff") == t).filter(pl.col("fips")==c)
    y_tc = x_tc.select(outcome_col).to_pandas().values
    x_tc = x_tc.select(feature_cols).with_columns(pl.lit(1).alias("Intercept")).to_pandas().values

    return x_tc, y_tc

def stage2_beta(stage1_beta_with_intercept, x_tc, y_tc, t, c, lambda_exp):
    lambda_reg = 10**lambda_exp

    # Define variable for the optimization (beta to be optimized)
    beta_tc = cp.Variable(stage1_beta_with_intercept.shape[0])
    
    predicted = cp.reshape(x_tc @ (beta_tc + stage1_beta_with_intercept), (2, 1))
    
    # Define the RMSE term: (1/sqrt(n)) * || X @ beta - y ||_2
    rmse_term = cp.norm(predicted - y_tc, 2)**2 / (y_tc.shape[0])
    
    # Define the regularization term: lambda * || beta - stage1_beta ||_1
    reg_term = lambda_reg * cp.norm(beta_tc, 1)
    
    # Objective function: minimize RMSE + lambda * regularization term
    objective = cp.Minimize(rmse_term + reg_term)
    
    # Problem setup
    problem = cp.Problem(objective)
    
    # Solve the problem
    problem.solve(), beta_tc.value

    # Save stage2_beta
    fname = f"./stage2_betas/lambda_exp={int(lambda_exp)}/stage2_beta_lambda_exp={int(lambda_exp)}_cutoff={t}_fips={c}.npy"
    tqdm.write("Saving {}".format(fname))
    np.save(fname, beta_tc.value)
    
    y_pred_fname = f"./stage2_betas/lambda_exp={int(lambda_exp)}/stage2_y_pred_exp={int(lambda_exp)}_cutoff={t}_fips={c}.npy"
    tqdm.write("Saving {}".format(y_pred_fname))
    np.save(y_pred_fname, x_tc @ beta_tc.value)


    return True

# Define a helper function for parallel execution
def parallel_stage2_beta(args):
    fname = f"./stage2_betas/lambda_exp={int(lambda_exp)}/stage2_beta_lambda_exp={int(lambda_exp)}_cutoff={t}_fips={c}.npy"
    if os.path.exists(fname):
        tqdm.write(f"{fname} exists. Skipping")
        return None
    try:
        res = stage2_beta(*args)
        return res
    except Exception as e:
        # If an error occurs, log the error and return an error message or None
        tqdm.write(f"Error for t={t}, c={c}, lambda_exp={lambda_exp}: {str(e)}")
        return None

# Function to parallelize across t, c
def parallelize_stage2_beta(stage1_beta_with_intercepts, combined_data=combined_data, t_values=cutoff_list, c_values=fips_list, lambda_exps=lambda_exps):
    # Create a list of arguments (t, c) combinations
    combined_data = combined_data.filter(pl.col("cutoff") in set(cutoff_list)).filter(pl.col("fips") in set(fips_list))
    tasks = [(stage1_beta_with_intercept, combined_data, t, c, lambda_exp) for t in t_values for c in c_values for lambda_exp in lambda_exps for stage1_beta_with_intercept in stage1_beta_with_intercepts]

    
    results = []
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Submit tasks and handle them
        futures = {executor.submit(parallel_stage2_beta, task): task for task in tasks}

        results = []
        # Collect results and handle any task that fails
        for future in concurrent.futures.as_completed(futures):
            task = futures[future]
            try:
                result = future.result()  # Get the result of the task
                results.append(result)
            except Exception as e:
                tqdm.write(f"Task {task} generated an exception: {e}")
                results.append(None)  # Append None for failed tasks
    
    return results


def task_generator(combined_data, t_values, c_values, lambda_exps):
    """
    Yields x_tc, y_tc, t, c, stage1_beta
    """
    for t in t_values:
        stage1_model = joblib.load("./stage1_lasso/best_lasso_model_cutoff={}.pkl".format(t))
        beta = stage1_model.coef_
        intercept = stage1_model.intercept_
        stage1_beta_with_intercept = np.concatenate((beta, [intercept]))

        for c in c_values:
            tqdm.write(f"Slicing (t={t}, c={c})")
            x_tc, y_tc = slice_data(combined_data, t, c)  # Perform slicing dynamically
            for lambda_exp in lambda_exps:
                yield (stage1_beta_with_intercept, x_tc, y_tc, t, c, lambda_exp)  # Yield each task as a tuple


def mp_parallelize_stage2_beta(combined_data=combined_data, t_values=cutoff_list, c_values=fips_list, lambda_exps=lambda_exps):
    # Create a list of arguments (t, c, lambda) combinations
    results = []
    # Create a multiprocessing pool
    with Pool(processes=cpu_count()) as pool:
        # Submit tasks to the pool and use tqdm for progress bar
        #tasks = task_generator(combined_data, t_values, c_values, lambda_exps)
        
        # Use pool.imap to process tasks generated by the generator dynamically
        for result in tqdm(pool.imap(parallel_stage2_beta, task_generator(combined_data, t_values, c_values, lambda_exps)), total=len(t_values) * len(c_values) * len(lambda_exps)):
            results.append(result)  # Collect results
            
    return results


NameError: name 'combined_data' is not defined

In [ ]:
results = mp_parallelize_stage2_beta()

Slicing (t=500, c=1001)


  0%|          | 0/31300 [00:00<?, ?it/s]

Saving ./stage2_betas/lambda_exp=1/stage2_beta_lambda_exp=1_cutoff=500_fips=1001.npy
Saving ./stage2_betas/lambda_exp=1/stage2_y_pred_exp=1_cutoff=500_fips=1001.npy
Slicing (t=500, c=1003)
Saving ./stage2_betas/lambda_exp=1/stage2_beta_lambda_exp=1_cutoff=500_fips=1003.npy
Saving ./stage2_betas/lambda_exp=1/stage2_y_pred_exp=1_cutoff=500_fips=1003.npy
Saving ./stage2_betas/lambda_exp=1/stage2_beta_lambda_exp=1_cutoff=500_fips=1005.npy
Saving ./stage2_betas/lambda_exp=1/stage2_y_pred_exp=1_cutoff=500_fips=1005.npy
Saving ./stage2_betas/lambda_exp=1/stage2_beta_lambda_exp=1_cutoff=500_fips=1007.npy
Saving ./stage2_betas/lambda_exp=1/stage2_y_pred_exp=1_cutoff=500_fips=1007.npy
Slicing (t=500, c=1005)
Slicing (t=500, c=1007)
Slicing (t=500, c=1009)
Saving ./stage2_betas/lambda_exp=1/stage2_beta_lambda_exp=1_cutoff=500_fips=1009.npy
Saving ./stage2_betas/lambda_exp=1/stage2_y_pred_exp=1_cutoff=500_fips=1009.npy
Error for t=600, c=2240, lambda_exp=1: Invalid dimensions (0, 471).
Saving ./st

In [ ]:
print(results)